# 🧑‍🏫 Make Virgo smarter on Kaggle (free GPU): checked examples from Bayon-1.0-27B

This runs the long **distill** step of `bash scripts/train_local.sh all` on Kaggle instead of your own GPU, so your server keeps serving Virgo AI while the data is made.

**Bayon-1.0-27B** answers thousands of generated problems: math, logic, exact instructions and other languages, each in **English and Khmer**. Every answer is checked with the test-set grader (the right final number, the right language, the format asked for), and only verified answers are kept. When one of its two answers is wrong, the pair goes to the DPO data.

Everything is saved to your private Hugging Face dataset **Data-1.0** as it goes (`distilled_smart_kaggle.jsonl` and `smart_pairs_kaggle.jsonl`, separate from your server's files, so both can run at the same time). Your server's training uses them automatically.

**First time only**
1. On Hugging Face, open [google/gemma-3-27b-it](https://huggingface.co/google/gemma-3-27b-it) and click **Agree and access**.
2. Kaggle → **Settings → Phone verification**.
3. **Add-ons → Secrets**: `HF_TOKEN` (a Hugging Face token with **write** access to the `virgoai` organization), ticked for this notebook.

**Run**
1. **Session options**: **Accelerator → GPU T4 x2**, **Internet → On**.
2. **Save Version → Save & Run All (Commit)**. It runs in the background for up to 11 hours, so you can close the tab.
3. Run it again (a new version) to make more: it continues where it stopped. Kaggle gives about 30 GPU hours a week, so about 2–3 runs.

Afterwards, on your server: `bash scripts/train_local.sh angkor 12b`, then `bash scripts/train_local.sh dpo angkor 12b`.

In [ ]:
# ⚙️ Settings
TARGET = 12000          # checked examples to make on Kaggle, in total across runs
HOURS = 11              # stop and save before Kaggle's 12-hour limit
VIRGO_HF_ORG = "virgoai"
TEACHER = "Bayon-1.0-27B"   # Virgo's smartest model teaches (plain Gemma 3 27B if it isn't found)

In [ ]:
# 1️⃣ Set up
import os, subprocess, sys, torch
assert torch.cuda.device_count() >= 2, "❌ Session options → Accelerator → GPU T4 x2 (27B needs both GPUs)"
print("✅ GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
WORK = "/kaggle/working"
os.chdir(WORK)
if not os.path.isdir("virgo-models/.git"):
    r = subprocess.run(["git", "clone", "-q", "https://github.com/nangmrr752/virgo-models"])
    assert r.returncode == 0, "❌ Can't download: Session options → Internet → On (needs phone verification)"
os.chdir(f"{WORK}/virgo-models")
subprocess.run(["git", "pull", "-q"])
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-train.txt"], check=True)
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"])  # conflicts with newer peft
from huggingface_hub import login, HfApi, hf_hub_download, snapshot_download
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    token = None
assert token, "❌ Add the HF_TOKEN secret (Add-ons → Secrets) and tick it for this notebook"
login(token=token)
os.environ["VIRGO_HF_ORG"] = VIRGO_HF_ORG
sys.path.append("scripts")
from names import resolve
api = HfApi()
DATA_REPO = resolve("Data-1.0", api) or f"{VIRGO_HF_ORG}/Data-1.0"
OUT, PAIRS = "chat/data/distilled_smart_kaggle.jsonl", "chat/dpo/smart_pairs_kaggle.jsonl"
os.makedirs("chat/dpo", exist_ok=True)
for path in (OUT, PAIRS):  # continue from earlier Kaggle runs
    try:
        hf_hub_download(DATA_REPO, os.path.basename(path), repo_type="dataset", local_dir=os.path.dirname(path))
        print("⬇️ Continuing:", path, sum(1 for _ in open(path, encoding="utf-8")), "lines")
    except Exception:
        print("Starting fresh:", path)
ADAPTER = ""
teacher_repo = resolve(TEACHER, api, need_file="adapter_config.json")
try:
    ADAPTER = snapshot_download(teacher_repo, local_dir="/kaggle/working/teacher")
except Exception as err:  # no Bayon yet: Gemma 3 27B itself teaches
    print(f"{TEACHER} not found ({err.__class__.__name__}): Gemma 3 27B teaches")
    teacher_repo = None
print("✅ Ready. Teacher:", teacher_repo or "google/gemma-3-27b-it", "· saving to", DATA_REPO)

def run(cmd):
    """Runs a step and shows its output here; on failure, shows the last lines (send them to Claude)."""
    import collections
    tail = collections.deque(maxlen=60)
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="")
        tail.append(line)
    if proc.wait() != 0:
        print("\n" + "=" * 60 + "\n❌ It stopped with an error. Send these last lines to Claude:\n" + "=" * 60)
        print("".join(tail))
        raise SystemExit(f"❌ {next((a for a in cmd if a.endswith('.py')), cmd[0])} failed (exit code {proc.returncode})")

In [ ]:
# 2️⃣ Make checked examples (saved to Hugging Face every 300; stops by itself after HOURS)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
run([sys.executable, "-u", "scripts/distill_smart.py", "--teacher", "google/gemma-3-27b-it", "--adapter", ADAPTER,
     "--target", str(TARGET), "--hours", str(HOURS), "--batch", "4", "--out", OUT, "--pairs", PAIRS, "--hf-repo", DATA_REPO])

In [ ]:
# 3️⃣ A few of the new examples (send some to Claude if anything looks off)
import json, random
rows = [json.loads(l) for l in open(OUT, encoding="utf-8")]
for r in random.sample(rows, min(6, len(rows))):
    m = r["messages"]
    print("🧑", m[1]["content"][:300]); print("🤖", m[2]["content"][:600], "\n")
pairs = sum(1 for _ in open(PAIRS, encoding="utf-8")) if os.path.exists(PAIRS) else 0
print(f"✅ {len(rows)} checked examples and {pairs} DPO pairs in {DATA_REPO}.")